# Code to retrieve absolute fingerprint features from SIRIUS

This notebook retrieves absolute indexes for fingerprint features present in a compound based on its SMILES representation.
It requires a file with only the desired SMILES to be fingerprinted as a tsv or csv in as input.
It creates a file containing the given SMILES and the retrieved absolute fingerprint feature indexes, 
and a version file that contains the information associated with each absolute index, including a description of the encoded substructure.

## 1. Imports

In [ ]:
import subprocess

## 2. Helper Function

In [ ]:
def run_sirius_fingerprinter(
    sirius_path: str,
    email: str,
    password: str,
    smiles_file: str,
    output_file: str,
    fp_version_file: str,
    charge: int = 1
):
    """
    Run SIRIUS fingerprinter from Python.

    Parameters:
        sirius_path (str): Path to the SIRIUS executable.
        email (str): SIRIUS login email.
        password (str): SIRIUS login password.
        smiles_file (str): Path to input SMILES file (TSV format). Can only contain SMILES strings.
        output_file (str): Path to output file for fingerprints.
        fp_version_file (str): Path to output file for fingerprint version info.
        charge (int): Ionization mode, use 1 (positive) or -1 (negative). Default is 1.
    """
    
    # === LOGIN TO SIRIUS ===
    login_process = subprocess.Popen(
        [sirius_path, 'login', '-u', email, '-p'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        stdin=subprocess.PIPE,
        text=True
    )
    stdout, stderr = login_process.communicate(input=password + '\n')
    
    if login_process.returncode != 0:
        raise RuntimeError(f"Login failed:\n{stderr}")

    # === RUN FINGERPRINTER ===
    command = [
        sirius_path,
        '-i', smiles_file,
        'fingerprinter',
        '--charge', str(charge),
        '-o', output_file,
        '-v', fp_version_file
    ]

    result = subprocess.run(command, capture_output=True, text=True)
    
    if result.returncode != 0:
        raise RuntimeError(f"Fingerprinter failed:\n{result.stderr}")
    
    print("SIRIUS fingerprinting completed successfully.")

## Run SIRIUS fingerprinter to get fingerprints

In [ ]:
#run SIRIUS fingerprinter to get fingerprints
run_sirius_fingerprinter(sirius_path="/path/to/sirius", email="your.email@example.com", password="your_password", smiles_file="your_file_with_only_SMILES.tsv", output_file="SMILES_with_abs_fps.tsv", fp_version_file="SIRIUS_version_file.tsv")